# Notebook 10: PPO Evaluation

**Purpose:** Evaluate the trained PPO controller agent and compare its performance metrics directly against the Fixed-Time traffic baseline controller.


In [1]:
import sys
import numpy as np
import json
from pathlib import Path
from stable_baselines3 import PPO

sys.path.append(str(Path("..").resolve()))
from backend.rl.environment import EcoTwinEnv
from backend.simulation.traffic_state import TrafficState
from backend.simulation.emission_collector import EmissionCollector


2026-08-21 12:03:34,163 [INFO] EcoTwin: Logging initialized successfully


2026-08-21 12:03:34,164 [INFO] EcoTwin: SUMO_HOME verified: C:\Program Files (x86)\Eclipse\Sumo\


### Run PPO Controller on Evaluation Scenario

In [2]:
env = EcoTwinEnv(config={
    "scenario": "evaluation",
    "gui": False,
    "step_length": 1.0,
    "max_steps": 1000
})

model = PPO.load("../models/ppo/best_model.zip")

obs, info = env.reset()
done = False
truncated = False

co2_steps = []
nox_steps = []
fuel_steps = []
wait_steps = []
speed_steps = []

print("Running PPO controller agent in SUMO...")
while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)
    
    # Track metrics
    emissions = EmissionCollector.get_system_emissions()
    co2_steps.append(emissions["co2"])
    nox_steps.append(emissions["nox"])
    fuel_steps.append(emissions["fuel"])
    
    vehicles = TrafficState.get_active_vehicles()
    if vehicles:
        avg_wait = np.mean([v["waiting_time"] for v in vehicles])
        avg_speed = np.mean([v["speed"] for v in vehicles])
    else:
        avg_wait = 0.0
        avg_speed = 0.0
        
    wait_steps.append(avg_wait)
    speed_steps.append(avg_speed)

env.close()


Running PPO controller agent in SUMO...


SUMO simulation closed.


### Compare Metrics: PPO vs Fixed-Time

In [3]:
# Load fixed time metrics
with open("../reports/fixed_time_metrics.json") as f:
    fixed_metrics = json.load(f)

ppo_metrics = {
    "total_co2_mg": float(np.sum(co2_steps)),
    "average_co2_mg_s": float(np.mean(co2_steps)),
    "total_nox_mg": float(np.sum(nox_steps)),
    "total_fuel_ml": float(np.sum(fuel_steps)),
    "average_waiting_time_s": float(np.mean(wait_steps)),
    "average_speed_kmh": float(np.mean(speed_steps))
}

print("PPO Agent Performance:")
print(json.dumps(ppo_metrics, indent=4))

# Calculate percentage improvement
improvements = {}
for k in fixed_metrics.keys():
    baseline_val = fixed_metrics[k]
    ppo_val = ppo_metrics[k]
    
    if "speed" in k:
        # For speed: Higher is better
        imp = ((ppo_val - baseline_val) / (baseline_val + 1e-5)) * 100
    else:
        # For delay, emissions, fuel: Lower is better
        imp = ((baseline_val - ppo_val) / (baseline_val + 1e-5)) * 100
    improvements[k] = imp

print("\nPercentage Improvements (Positive is improvement):")
for k, v in improvements.items():
    print(f"{k}: {v:+.2f}%")

with open("../reports/ppo_training/improvements.json", "w") as f:
    json.dump({"ppo_metrics": ppo_metrics, "improvements": improvements}, f, indent=4)


PPO Agent Performance:
{
    "total_co2_mg": 542580224.2693118,
    "average_co2_mg_s": 542580.2242693118,
    "total_nox_mg": 2562288.1021477506,
    "total_fuel_ml": 178457228.52703708,
    "average_waiting_time_s": 100.11088408105768,
    "average_speed_kmh": 6.747392997563874
}

Percentage Improvements (Positive is improvement):
total_co2_mg: -301.11%
average_co2_mg_s: -301.11%
total_nox_mg: -322.22%
total_fuel_ml: -306.38%
average_waiting_time_s: -8404.99%
average_speed_kmh: -83.23%
